In [1]:
import h5py
import numpy as np
import joblib
import torch

In [2]:
print(torch.cuda.is_available())

True


In [3]:
DS_ROOT = '/home/mcarroll/Documents/cd-2/VideoMimic/PDP/data/phc_vidmimic_0.05'


In [4]:
meta_data = joblib.load(f'{DS_ROOT}/phc_act_amass_train_upright_metadata.pkl')
failed_keys = joblib.load(f'{DS_ROOT}/failed.pkl')

In [5]:
print(meta_data.keys())

motion_lengths = np.concatenate( [ml for ml in meta_data['motion_lengths']])
keys_names = np.concatenate( [kn for kn in meta_data['key_names']])

print(len(motion_lengths))
print(sum(motion_lengths))
print(len(keys_names))
print(keys_names)
num_motions = len(keys_names)
motion_starts = np.cumsum(motion_lengths)
motion_starts = np.insert(motion_starts, 0, 0)
motion_starts = motion_starts[:-1]
print(motion_starts[:5])

dict_keys(['key_names', 'motion_lengths', 'running_mean', 'config', 'exclude_ids'])
192470
25875916
192470
['20221013_3-10_500_batch01hand_static_highSchoolGym/seq_000350/rp_celina_posed_005'
 '20221013_3-10_500_batch01hand_static_highSchoolGym/seq_000350/rp_beatrice_posed_036'
 '20221013_3-10_500_batch01hand_static_highSchoolGym/seq_000350/rp_janna_posed_037'
 ...
 '20221019_3-8_250_highbmihand_orbit_stadium/seq_000215/male_35_nl_5786'
 '20221019_3-8_250_highbmihand_orbit_stadium/seq_000215/male_39_nl_6338'
 '20221019_3-8_250_highbmihand_orbit_stadium/seq_000215/female_31_nl_1440']
[  0 136 272 408 544]


In [8]:
print(len(failed_keys))
print(failed_keys)

2243
['20221011_1_250_batch01hand_closeup_suburb_c/seq_000240/rp_christine_posed_025'
 '20221011_1_250_batch01hand_closeup_suburb_d/seq_000078/rp_emma_posed_002'
 '20221012_1_500_batch01hand_closeup_highSchoolGym/seq_000267/rp_janna_posed_044'
 ... '20221017_3_1000_batch01hand/seq_000537/rp_aneko_posed_024'
 '20221024_3-10_100_batch01handhair_static_highSchoolGym/seq_000041/rp_fernanda_posed_028'
 '20221017_3_1000_batch01hand/seq_000995/rp_christine_posed_013']


In [9]:
dataset_path = f'{DS_ROOT}/phc_act_amass_train_upright.h5'
with h5py.File(dataset_path, 'r') as hdf5_file:
    # Get the total size of the dataset
    for key in hdf5_file.keys():
        print(f"- {key}")
    dataset_size = len(hdf5_file['clean_action']) 

    assert dataset_size == sum(motion_lengths)
    # Load the `reset` boolean array
    reset = hdf5_file['reset'][:]
    action  =  hdf5_file['clean_action'][:]
    obs  =  hdf5_file['pdp_obs'][:]



- clean_action
- pdp_obs
- pdp_ref
- reset


KeyboardInterrupt: 

In [ ]:

exclude_ids = []
exclude_indicies = []

obs_dim = obs.shape[-1]
action_dim = action.shape[-1]

count_obs = 0
mean_obs = np.zeros(obs_dim, dtype=obs.dtype)
M2_obs = np.zeros(obs_dim, dtype=obs.dtype)
# Initialize min and max as before
obs_min = np.full(obs_dim, np.inf, dtype=obs.dtype)
obs_max = np.full(obs_dim, -np.inf, dtype=obs.dtype)


count_action = 0
mean_action = np.zeros(action_dim, dtype=obs.dtype)
M2_action = np.zeros(action_dim, dtype=obs.dtype)
# Initialize min and max as before
action_min = np.full(action_dim, np.inf, dtype=obs.dtype)
action_max = np.full(action_dim, -np.inf, dtype=obs.dtype)

for m_i in range(num_motions):

    ml = motion_lengths[m_i]
    start_idx, end_idx = motion_starts[m_i], motion_starts[m_i]+ml


    if keys_names[m_i] in failed_keys:
        exclude_ids.append(m_i)
        exclude_indicies.extend(range(start_idx, end_idx))
        continue
    

    m_r = reset[start_idx:end_idx]
    m_a = action[start_idx:end_idx]
    m_o = obs[start_idx:end_idx]

    has_no_reset = (np.sum(m_r) == 0)
    if has_no_reset:
        exclude_ids.append(m_i)
        exclude_indicies.extend(range(start_idx, end_idx))
        continue

    is_last_a_reset = (m_r[-1] == 1)
    try:
        assert is_last_a_reset, 'Last index is not Reset'
    except:
        exclude_ids.append(m_i)
        print(m_i)
        continue

    has_only_one_reset = (np.sum(m_r) == 1)
    assert has_only_one_reset, 'More the one Reset in data'

    #OBS -------------- Update min/max
    obs_min = np.minimum(obs_min, m_o.min(axis=0))
    obs_max = np.maximum(obs_max, m_o.max(axis=0))

    # Update mean and M2 using Welford's algorithm -------------
    chunk_count = m_o.shape[0]
    
    delta = m_o - mean_obs
    new_mean_obs = mean_obs + np.sum(delta, axis=0) / (count_obs + chunk_count)
    
    delta2 = m_o - new_mean_obs
    M2_obs += np.sum(delta * delta2, axis=0)
    
    mean_obs = new_mean_obs
    count_obs += chunk_count


    #OBS -------------- Update min/max ----------------
    action_min = np.minimum(action_min, m_a.min(axis=0))
    action_max = np.maximum(action_max, m_a.max(axis=0))

    # Update mean and M2 using Welford's algorithm
    chunk_count = m_o.shape[0]
    
    delta = m_a - mean_action
    new_mean_action = mean_action + np.sum(delta, axis=0) / (count_action + chunk_count)
    
    delta2 = m_a - new_mean_action
    M2_action += np.sum(delta * delta2, axis=0)
    
    mean_action = new_mean_action
    count_action += chunk_count



all_indices = np.arange(len(action))
indices_to_keep = np.setdiff1d(all_indices, exclude_indicies)

std_obs = np.sqrt(M2_obs / count_obs)
std_action = np.sqrt(M2_action / count_action)


5145
12356
54440
65228
157108


In [ ]:
import sys
import os


module_path = os.path.abspath('/home/mcarroll/Documents/cd-2/VideoMimic/PDP')

# Insert the path at the beginning of the list
sys.path.insert(0, module_path)
from pdp.utils.normalizer import LinearNormalizer

In [ ]:
data = {
    'obs': {
        'min':obs_min,
        'max':obs_max,
        'mean':mean_obs,
        'std':std_obs,
    },
    'action': {
        'min':action_min,
        'max':action_max,
        'mean':mean_action,
        'std':std_action,
    },
}

normalizer = LinearNormalizer()
normalizer.fit_implicit(data=data, mode='limits')


{'min': array([-1.15021482e-01, -1.14825442e-01, -1.15020059e-01, -4.90020990e-01,
       -4.85829711e-01, -4.91332233e-01, -8.79895270e-01, -8.84981811e-01,
       -8.88764143e-01, -9.84661579e-01, -9.68966007e-01, -1.00755191e+00,
       -1.13097742e-01, -1.13025457e-01, -1.13098584e-01, -4.96121705e-01,
       -4.96455789e-01, -4.97563869e-01, -8.85012984e-01, -8.90012205e-01,
       -8.96753192e-01, -9.92689133e-01, -1.00867105e+00, -1.01584435e+00,
       -1.12238429e-01, -1.09032974e-01, -1.12178050e-01, -2.46043786e-01,
       -2.44292259e-01, -2.29800045e-01, -3.00358385e-01, -3.02657396e-01,
       -2.85963953e-01, -5.14329493e-01, -5.14213443e-01, -4.89577234e-01,
       -5.75257838e-01, -5.85948944e-01, -5.64485013e-01, -4.34267074e-01,
       -4.36586380e-01, -3.97090316e-01, -5.10465026e-01, -5.25579572e-01,
       -4.78842258e-01, -7.30400801e-01, -7.09203064e-01, -7.19623744e-01,
       -8.47770035e-01, -8.93025041e-01, -9.52179134e-01, -9.29754376e-01,
       -9.5954585

In [ ]:
normalizer_state = normalizer.state_dict()

# Save the state dictionary to a file
torch.save(normalizer_state, f'{DS_ROOT}/normalizer_params.pt')

In [ ]:
meta_data['exclude_ids'] = exclude_ids
joblib.dump(meta_data, f'{DS_ROOT}/phc_act_amass_train_upright_metadata.pkl')

['/home/mcarroll/Documents/cd-2/VideoMimic/PDP/data/phc_vidmimic_0.05/phc_act_amass_train_upright_metadata.pkl']

# Build the image mappings and preproc embeddings

In [6]:
# load the image phc dataset

phc_image_ds = joblib.load(f'/home/mcarroll/Documents/cd-2/VideoMimic/PHC/data/amass/amass_train_upright_vidmimic.pkl')

In [7]:
vidmimic_image_mappings = {}
for mk in keys_names:
    seq = phc_image_ds[mk]
    vidmimic_image_mappings[mk] = seq['images']

joblib.dump(vidmimic_image_mappings, f'{DS_ROOT}/vidmimic_image_mappings.pkl')


['/home/mcarroll/Documents/cd-2/VideoMimic/PDP/data/phc_vidmimic_0.05/vidmimic_image_mappings.pkl']

In [6]:
import torch
from PIL import Image
import requests
from transformers import AutoImageProcessor, AutoModel
import cv2
import torch
from transformers.image_utils import load_image

In [7]:
from pdp.lora_model import load_dinov2


dino_fn, dino_dim = load_dinov2(device='cuda')

In [8]:
torch.cuda.empty_cache()


In [ ]:
import joblib
import os
image_map = joblib.load(f'{DS_ROOT}/vidmimic_image_mappings.pkl')
IMG_DS_ROOT = '/home/mcarroll/Documents/BEDLAM/crops'

In [12]:


embs = {}

from tqdm import tqdm

MAX_BS = 512

with torch.no_grad():
    for k, v in tqdm(image_map.items(), desc="Processing image mappings"):
        dino_embs_list = []
        # Process in batches of MAX_BS
        for i in range(0, len(v), MAX_BS):
            batch_files = v[i:i+MAX_BS]
            image_list = [load_image(f'{IMG_DS_ROOT}/{img_path}') for img_path in batch_files]
            batch_embs = dino_fn(image_list)
            dino_embs_list.append(batch_embs.cpu())
        # Concatenate all batches for this key
        dino_embs = torch.cat(dino_embs_list, dim=0)
        embs[k] = dino_embs.numpy()

joblib.dump(embs, f'{DS_ROOT}/vidmimic_image_emb_mappings.pkl')


Processing image mappings: 100%|██████████| 38259/38259 [3:58:38<00:00,  2.67it/s]  


['/home/mcarroll/Documents/cd-2/VideoMimic/PDP/data/phc_vidmimic_0.05/vidmimic_image_emb_mappings.pkl']

# Reformat the h5 file - debug


In [1]:
import h5py

In [3]:
with h5py.File("/home/mcarroll/Documents/cd-2/VideoMimic/PDP/data/phc_vidmimic_0.05/phc_act_amass_train_upright.h5", "r") as f_in, h5py.File("/home/mcarroll/Documents/cd-2/VideoMimic/PDP/data/phc_vidmimic_0.05/phc_act_amass_train_upright_v2.h5", "w") as f_out:
    for k, dset in f_in.items():
        if k in ["pdp_obs", "clean_action", "pdp_ref"]:
            shape = dset.shape
            dtype = dset.dtype
            chunk_len = 64  # or your horizon
            chunk_shape = (min(chunk_len, shape[0]),) + shape[1:]

            new_dset = f_out.create_dataset(
                k,
                shape=shape,
                dtype=dtype,
                chunks=chunk_shape,
                compression="lzf",
                shuffle=True
            )

            # Read everything into numpy first, then write
            new_dset[:] = dset[:]